# Sahformer — Kaggle training

Clock-aware Chessformer (faithful Maia-3 5M backbone + our time layer) on Kaggle's GPU.

**Before running — open the right-hand Settings panel and set:**
- **Accelerator → GPU (T4 x2 or P100)**
- **Internet → On**  (needed to clone the repo + stream the games)
- **Persistence → Files only**  (so your trained model survives between sessions)

Then edit `REPO_URL` below and run top to bottom. This is a **big run**: 5M positions
(streamed, so memory stays flat) training a ~14M-param model — roughly **20-30 min to build
the data + a few hours to train**. Best done as a background run: **Save Version → Save & Run
All (Commit)**, then close the tab. The model saves to `/kaggle/working/sahformer_ckpts/full/`
— the **Save & download** cell zips it so you can pull it to your PC. **Download it after
training so you never lose it.**

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/slobaspid/sah-transformer.git"  # <-- EDIT if different
import os
if not os.path.isdir("sah-transformer"):
    !git clone "$REPO_URL"
%cd sah-transformer
!git pull --ff-only || true

In [ ]:
# 2) Deps + GPU check (torch ships with Kaggle)
!pip -q install python-chess zstandard
import sys; sys.path.insert(0, ".")
import torch
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| cuda", ok, "|",
      torch.cuda.get_device_name(0) if ok else "NO GPU -> Settings > Accelerator > GPU")

## 3) Build training shards (streamed from Lichess, chunked)

Streams a 2017-04+ month (these have `%clk` clocks) and early-stops. It writes the data in
**chunks**, so memory stays flat — with Kaggle's ~30 GB you can pull a million positions.
Output is several `data/shard0000.npz`… files; training reads them all.
`BALANCE=False` keeps more data (Elo-skewed); `True` balances each chunk but shrinks it.

In [ ]:
# 3) Fetch + build (chunked, memory stays flat) — this takes ~20-30 min for 5M
URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst"
MAX_POSITIONS = 5_000_000   # big run; training streams these so RAM stays flat
BALANCE = False

import glob
from sahformer.download import stream_games_from_url
from sahformer.dataset_build import build_shards, records_from_games

build_shards(
    records_from_games(stream_games_from_url(URL)),
    "data", chunk_positions=200000, max_positions=MAX_POSITIONS,
    balance=BALANCE, progress_every=25000)
DATA_SHARDS = sorted(glob.glob("data/*.npz"))
print(len(DATA_SHARDS), "shards")

## 4) Checkpoint location

No Google Drive here — Kaggle keeps whatever you write under `/kaggle/working/`. You can
download files from the **Output** tab, or **Save Version** to persist them with the notebook.

In [ ]:
OUT = "/kaggle/working/sahformer_ckpts"
import os; os.makedirs(OUT, exist_ok=True)
print("checkpoints ->", OUT)

## 5) Train the clock-aware model (`full`) on GPU with AMP

Writes `best.pt` / `last.pt` straight to `/kaggle/working` (fast local disk). Bump `max_steps`
for a longer run; watch the loss fall.

In [ ]:
import glob
from sahformer.model.config import ModelConfig
from sahformer.training.build import build_model
from sahformer.training.loop import TrainConfig, train
DATA_SHARDS = sorted(glob.glob("data/*.npz"))

# bigger model (~14M params vs the 5.9M default). The checkpoint stores this size,
# so the local viewer/engine rebuild it automatically.
MODEL = ModelConfig(dim_vit=384, num_blocks=10)
print(sum(p.numel() for p in build_model("full", MODEL).parameters())/1e6, "M params")

# stream=True keeps RAM flat over the 5M positions; this run takes a few hours on a T4.
cfg = TrainConfig(mode="full", max_steps=30000, warmup_steps=1500, batch_size=512,
                  lr=3e-4, amp=True, stream=True, device="cuda", out_dir=f"{OUT}/full",
                  log_every=200, ckpt_every=2000)
res = train(cfg, DATA_SHARDS, model_cfg=MODEL)
print("full best_total:", res["best"], "| saved to", f"{OUT}/full")

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; xs = [r["step"] for r in h]
for key in ("total", "policy", "time"):
    plt.plot(xs, [r[key] for r in h], label=key)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("full — losses"); plt.show()
plt.plot(xs, [r["move_acc"] for r in h]); plt.xlabel("step"); plt.ylabel("move_acc")
plt.title("full — move accuracy (sanity metric)"); plt.show()

## 5b) SAVE & DOWNLOAD your model  ⬇️  (do this now!)

This zips your trained model so you can pull it to your PC. Right after it runs, open the
**Output** panel on the right, find `model_full.zip`, and click **Download**. That's your own
copy — it can't be lost to a Kaggle hiccup, and it's exactly the file the desktop chess engine
needs.

In [ ]:
import os, shutil
CKPT_DIR = f"{OUT}/full"
print("Your model is saved here:")
for f in sorted(os.listdir(CKPT_DIR)):
    p = os.path.join(CKPT_DIR, f)
    print(f"   {p}   ({os.path.getsize(p)/1e6:.1f} MB)")
shutil.make_archive("/kaggle/working/model_full", "zip", CKPT_DIR)
print("\n==> DOWNLOAD THIS FILE: /kaggle/working/model_full.zip")
print("    (right-hand Output panel -> model_full.zip -> Download)")
print("    Also confirm Settings > Persistence = 'Files only' to keep it on Kaggle too.")

## 6) Optional: baseline (clock-blind) for the later ablation comparison

In [ ]:
cfg_b = TrainConfig(mode="baseline", max_steps=30000, warmup_steps=1500, batch_size=512,
                    lr=3e-4, amp=True, stream=True, device="cuda", out_dir=f"{OUT}/baseline",
                    log_every=200, ckpt_every=2000)
res_b = train(cfg_b, DATA_SHARDS, model_cfg=MODEL)
print("baseline best:", res_b["best"], "| full best:", res["best"])

## Done

Checkpoints are in `/kaggle/working/sahformer_ckpts/`. Download them from the **Output** tab,
or **Save Version** to keep them with the notebook. The proper test stage (unseen games,
timing calibration, sampled play) is the next plan — this run just trains the model for real.